データ元: https://www.e-stat.go.jp/dbview?sid=0003449073

In [1]:
import pandas as pd
import datetime
from plotly import graph_objects as go
from plotly.offline import init_notebook_mode, iplot
from plotly.subplots import make_subplots
init_notebook_mode(connected=True)
import plotly.io as pio
pio.renderers.default = 'notebook'

In [2]:
df = pd.read_csv(
    '.ignored/data.csv',
    skiprows=10,
    thousands=','
)
df["時間軸（月次）"] = pd.to_datetime(df["時間軸（月次）"].apply(lambda x: datetime.datetime.strptime(x, '%Y年%m月')))
    
df = df.groupby([
    "地方出入国在留管理局・支局",
    "在留資格審査",
    "時間軸（月次）"
])[[
    "/在留資格審査の受理・処理","受理_総数","受理_旧受","受理_新受","既済_総数","既済_許可","既済_不許可","既済_その他"
]].sum().xs(
    "東京出入国在留管理局管内",
    level="地方出入国在留管理局・支局"
).xs(
    "永住",
    level="在留資格審査"
)

def unstring(x: str | float | int) -> str:
    if isinstance(x, (float, int)):
        return float(x)
    if x == '***':
        return 0.
    return float(x.replace(',', ''))

for c in df.columns:
    df[c] = df[c].apply(unstring)

永住権申請の結果を今待っているところで、なんやかんや色々気になっちゃうので一目で入管のメトリクスが見れるようにしてみました！☆

注意点:

- ここに表示されているのは東京入管のみのデータです。地方管理局の分は含まれていません。

<style>
    .main-container {
        max-width: 1500px;
    }
</style>

## 申請バックログ

毎月の受理中（未処理）、つまり入管がまだ手を付けていない申請。

- 減少傾向 -> 入管の仕事が比較的速い、新規処理が少ない、人手が余っている
- 増加傾向 -> 入管の仕事が比較的遅い、新規処理が多い、人手が足りていない

In [18]:
fig = go.Figure(layout={ 'title': '申請バックログ', 'showlegend': True, 'legend': { 'orientation': 'h' } })
fig.add_trace(go.Scatter(x=df.index, y=df['受理_総数'], name='受理_総数'))
fig.show()

## 受給

毎月の新規申請と処理済み申請。

- 新規申請の方が多い -> 申請バックログが増える
- 新規申請の方が少ない -> 申請バックログが減る

In [19]:
fig = go.Figure(layout={
    'title': '受給',
    'yaxis': {
        'range': [0, 8000]
    },
    'showlegend': True,
    'legend': { 'orientation': 'h' }
})
fig.add_trace(go.Scatter(x=df.index, y=df['既済_総数'], name='既済_総数', line={ 'dash': 'dash' }))
fig.add_trace(go.Scatter(x=df.index, y=df['受理_新受'], name='受理_新受', line={ 'dash': 'dot' }))
fig.show()

## 処理能力

毎月の受給の差分。

- 正の値 -> 処理済み申請の方が多い（バックログが減る、入管の仕事が比較的速い）
- 負の値 -> 新規申請の方が多い（バックログが増える、入管の仕事が比較的遅い）

In [22]:
fig = go.Figure(layout={
    'title': '処理能力',
    'yaxis': {
        'range': [-5_000, 5_000]
    },
    'showlegend': True,
    'legend': { 'orientation': 'h' }
})
fig.add_trace(go.Bar(x=df.index, y=df['既済_総数'] - df['受理_新受'], name='既済_総数 - 受理_新受'))
fig.show()

## 許可率

毎月の、処理済み申請総数に対する許可割合と不許可割合。

In [25]:
fig = go.Figure(layout={
    'title': '許可率',
    'yaxis': {
        'range': [0.0, 1.0]
    },
    'showlegend': True,
    'legend': { 'orientation': 'h' }
})
fig.add_trace(go.Scatter(x=df.index, y=df["既済_許可"] / df['既済_総数'], name='既済_許可 ÷ 既済_総数'))
fig.add_trace(go.Scatter(x=df.index, y=df["既済_不許可"] / df['既済_総数'], name='既済_不許可 ÷ 既済_総数'))
fig.show()

## 許可数

毎月の、処理された申請の総数と許可された申請の数を両方示したもの。

総数が増えているのに許可申請数が横ばいになっているということは色んな仮設で説明できますが、追加のデータがないと検証できそうにありません。

1. 許可基準は変っておらず、基準不適合の申請だけが増えている。
2. 基準適合の申請が増えているのと同時に、基準が厳しくなっている。
3. 基準適合の申請が減っていると同時に、基準が緩くなっている。

In [27]:
fig = go.Figure(layout={
    'title': '許可数',
    'showlegend': True,
    'legend': { 'orientation': 'h' }
})
fig.add_trace(go.Scatter(x=df.index, y=df['既済_総数'], name='既済_総数'))
fig.add_trace(go.Bar(x=df.index, y=df['既済_許可'], name='既済_許可'))
fig.show()